# 07 — XGBoost Deep Dive (FD001)

**Phase 6 (continued) — hyperparameter tuning + explainability groundwork for the strongest tree-based baseline.**

Goals:
- Randomized hyperparameter search
- Compare tuned vs. default XGBoost
- Feature importance

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from xgboost import XGBRegressor

from src.config.config import PROCESSED_DATA_DIR, RANDOM_STATE
from src.models.evaluate import compute_metrics
from src.models.train import save_model

VARIANT = 'FD001'
train_fe = pd.read_parquet(PROCESSED_DATA_DIR / f'train_{VARIANT}_features.parquet')
val_fe = pd.read_parquet(PROCESSED_DATA_DIR / f'val_{VARIANT}_features.parquet')
with open(PROCESSED_DATA_DIR / f'selected_features_{VARIANT}.json') as f:
    selected_features = json.load(f)['selected_features']

X_train, y_train = train_fe[selected_features], train_fe['RUL_clipped']
X_val, y_val = val_fe[selected_features], val_fe['RUL_clipped']

## Randomized hyperparameter search
Uses `GroupKFold` on `unit_number` (engine ID) so cross-validation folds never split a single engine's rows across train/test within a fold — keeping the same no-leakage guarantee as the rest of this project.

In [ ]:
param_distributions = {
    'n_estimators': randint(100, 300),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.03, 0.17),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
}

group_kfold = GroupKFold(n_splits=3)
groups = train_fe['unit_number']

# NOTE: n_jobs=-1 on the OUTER search only — the inner XGBRegressor
# is single-threaded (n_jobs=1) to avoid CPU oversubscription
# (nested parallelism), which is what makes this search fast.
search = RandomizedSearchCV(
    XGBRegressor(random_state=RANDOM_STATE, n_jobs=1),
    param_distributions=param_distributions,
    n_iter=8,
    scoring='neg_root_mean_squared_error',
    cv=list(group_kfold.split(X_train, y_train, groups=groups)),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print('Best CV RMSE:', -search.best_score_)

## Compare tuned vs. default XGBoost on validation engines

In [ ]:
default_model = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)
tuned_model = search.best_estimator_

default_metrics = compute_metrics(y_val, default_model.predict(X_val))
tuned_metrics = compute_metrics(y_val, tuned_model.predict(X_val))

pd.DataFrame({'default': default_metrics, 'tuned': tuned_metrics}).T

## Feature importance (tuned model)

In [ ]:
importances = pd.Series(
    tuned_model.feature_importances_, index=selected_features
).sort_values(ascending=False).head(20)

importances.plot(kind='barh', figsize=(8, 8), color='teal')
plt.gca().invert_yaxis()
plt.title('Top 20 XGBoost feature importances — FD001')
plt.tight_layout(); plt.show()

## Save the tuned model as the project's XGBoost baseline
Overwrites the default-hyperparameter XGBoost saved in notebook 06 with this tuned version, since it's expected to perform at least as well on the validation set.

In [ ]:
save_model(tuned_model, 'xgboost', selected_features, variant=VARIANT)
print('Saved tuned XGBoost model to models/rul/xgboost_FD001.joblib')

## Next
Continue to **08_lstm_gru_model.ipynb** for Phase 7: sequence models (LSTM / GRU / Temporal CNN / Transformer), compared against this tuned XGBoost baseline.